# Plaque Detection — Auswertung der Labeling-Ergebnisse

Lädt `labeling_results.csv` (manuelle TP/FN/FP/TN-Klassifikation) und
`yolo_detections_cache.json` (Modell-Detektionen), plottet Confusion Matrix
und Metriken, und zeigt alle geflaggten ("interessanten") Bilder mit
manueller Annotation (rot) und Modell-Detektion (grün) nebeneinander.

Nutzt `evaluate_labeling.py` aus dem selben Ordner für die Kennzahlen.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evaluate_labeling import evaluate, load_detections

INPUT_DIR = Path("train")

# Validated palette (design-system reference values - see dataviz skill)
GOOD = "#0ca30c"      # status: correct (TP, TN)
SERIOUS = "#ec835a"   # status: false positive
CRITICAL = "#d03b3b"  # status: false negative (worse in a screening context)
BLUE = "#2a78d6"      # categorical slot 1
ORANGE = "#eb6834"    # categorical slot 2
INK = "#0b0b0b"
SECONDARY_INK = "#52514e"
MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SURFACE = "#fcfcfb"


def style_axes(ax):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(MUTED)
    ax.tick_params(colors=SECONDARY_INK, length=0)
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=1, zorder=0)
    ax.set_axisbelow(True)

## Confusion Matrix & Metriken

In [ ]:
report, df = evaluate()
for k, v in report.items():
    print(f"{k:42s} {v:.3f}" if isinstance(v, float) else f"{k:42s} {v}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

categories = ["TP", "FN", "FP", "TN"]
values = [report["TP"], report["FN (total)"], report["FP (total)"], report["TN"]]
colors = [GOOD, CRITICAL, SERIOUS, GOOD]

bars = ax.bar(categories, values, color=colors, width=0.55, zorder=3)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + max(values) * 0.02, str(v),
            ha="center", va="bottom", fontsize=11, color=INK)

ax.set_ylabel("Anzahl Bilder", color=SECONDARY_INK, fontsize=10)
ax.set_title("Confusion Matrix — Plaque Detection", color=INK, fontsize=13, pad=14)
style_axes(ax)
plt.tight_layout()
plt.savefig("analysis_confusion_matrix.png", dpi=150, facecolor=SURFACE)
plt.show()

In [ ]:
metric_names = ["Precision", "Recall / Sensitivity", "Specificity", "F1", "Accuracy"]
short_names = ["Precision", "Recall", "Specificity", "F1", "Accuracy"]
metric_values = [report[m] for m in metric_names]

fig, ax = plt.subplots(figsize=(7, 4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
bars = ax.bar(short_names, metric_values, color=BLUE, width=0.5, zorder=3)
for bar, v in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}",
            ha="center", va="bottom", fontsize=11, color=INK)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Wert", color=SECONDARY_INK, fontsize=10)
ax.set_title("Modell-Metriken", color=INK, fontsize=13, pad=14)
style_axes(ax)
plt.tight_layout()
plt.savefig("analysis_metrics.png", dpi=150, facecolor=SURFACE)
plt.show()

## Domain-Shift: Miss-Rate nach Bildbereich

Bilder 1–276 vs. 277–552 (siehe Diskussion in der vorherigen Session) —
zeigt, ob die verpassten Plaques systematisch in einem Teil des Datensatzes
konzentriert sind.

In [ ]:
dets = load_detections()
df["det_present"] = df["image"].map(lambda n: bool(dets.get(n, {}).get("boxes")))
df["n"] = df["image"].str.replace(".png", "", regex=False).astype(int)

pos = df[df["has_label"] == "yes"].copy()
pos["is_fn"] = pos["result"].isin([0, 7])

low = pos[pos["n"] < 277]
high = pos[pos["n"] >= 277]
fn_rates = [low["is_fn"].mean(), high["is_fn"].mean()]
labels = [f"Bilder 1\u2013276\n(n={len(low)})", f"Bilder 277\u2013552\n(n={len(high)})"]

fig, ax = plt.subplots(figsize=(5, 4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
bars = ax.bar(labels, fn_rates, color=[BLUE, ORANGE], width=0.5, zorder=3)
for bar, v in zip(bars, fn_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.0%}",
            ha="center", va="bottom", fontsize=11, color=INK)

ax.set_ylim(0, 1.0)
ax.set_ylabel("FN-Rate (verpasste Plaques)", color=SECONDARY_INK, fontsize=10)
ax.set_title("Miss-Rate nach Bildbereich (Domain-Shift)", color=INK, fontsize=13, pad=14)
style_axes(ax)
plt.tight_layout()
plt.savefig("analysis_domain_shift.png", dpi=150, facecolor=SURFACE)
plt.show()

## Geflaggte ("interessante") Bilder

Rote Maske = manuelle Ground-Truth-Annotation, grüne Box(en) = Modell-Detektion
(mit Confidence).

In [ ]:
def build_overlay(name, alpha=0.4):
    base = cv2.imread(str(INPUT_DIR / name))
    base = cv2.cvtColor(base, cv2.COLOR_BGR2RGB).astype(np.float32)

    labeled_path = INPUT_DIR / f"{Path(name).stem}_labeled.png"
    if labeled_path.exists():
        mask = cv2.imread(str(labeled_path), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            if mask.shape[:2] != base.shape[:2]:
                mask = cv2.resize(mask, (base.shape[1], base.shape[0]))
            alpha_mask = (mask > 127).astype(np.float32) * alpha
            red = np.zeros_like(base)
            red[..., 0] = 255
            for c in range(3):
                base[..., c] = base[..., c] * (1 - alpha_mask) + red[..., c] * alpha_mask

    overlay = base.astype(np.uint8).copy()
    det = dets.get(name, {})
    for (x1, y1, x2, y2), conf in zip(det.get("boxes", []), det.get("confs", [])):
        cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(overlay, f"{conf:.2f}", (int(x1), max(int(y1) - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
    return overlay


result_labels = {0: "FN", 1: "TP", 2: "FP", 7: "Beides falsch"}

flagged = df[df["flag"] == "yes"].sort_values("n")
print(f"{len(flagged)} geflaggte Bilder")

n_cols = 3
n_rows = int(np.ceil(len(flagged) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), facecolor=SURFACE)
axes = np.atleast_1d(axes).flatten()

for ax, (_, row) in zip(axes, flagged.iterrows()):
    img = build_overlay(row["image"])
    ax.imshow(img)
    ax.axis("off")
    res = result_labels.get(row["result"], "-") if pd.notna(row["result"]) else "-"
    ax.set_title(f"{row['image']}  |  {res}", fontsize=10, color=INK)

for ax in axes[len(flagged):]:
    ax.axis("off")

fig.suptitle("Geflaggte Bilder \u2014 rote Maske = manuelles Labeling, gr\u00fcne Box = Modell-Detektion",
             fontsize=12, color=INK, y=1.02)
plt.tight_layout()
plt.savefig("analysis_flagged_images.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.show()